In [1]:
!pip install evaluate bert-score
!pip install sacrebleu
!pip install unbabel-comet

!pip install --upgrade protobuf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 8.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 7.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.5/55.5 kB 3.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 6.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.0/91.0 kB 12.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 101.4/101.4 kB 15.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 39.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 142.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 295.2/295.2 kB 42.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 852.4/852.4 kB 96.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 327.1/327.1 kB 9.5 MB/s eta 0:00:00
  Attempting uninstall: protobuf
    Found existing installation: protobuf 4.25.9
    Uninstalling protobuf-4.25.9:
      Successfully uninstalled protobuf-4.25.9
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
unbabel-comet 2.2.7 requires protobuf<5.0.0,>=4.24.4, but you have protobuf 7.35.1 which is incompatible.
google-ai-generativelanguage 0.6.15 requires protobuf!=4.21.0,!=4.21.1,!=4.21.2,!=4.21.3,!=4.21.4,!=4.21.5,<6.0.0dev,>=3.20.2, but you have protobuf 7.35.1 which is incompatible.
grpcio-status 1.71.2 requires protobuf<6.0dev,>=5.26.1, but you have protobuf 7.35.1 which is incompatible.
google-cloud-discoveryengine 0.13.12 requires protobuf!=4.21.0,!=4.21.1,!=4.21.2,!=4.21.3,!=4.21.4,!=4.21.5,<7.0.0,>=3.20.2, but you have protobuf 7.35.1 which is incompatible.
google-cloud

# **PREPROCESAMIENTO**

Se usa los archivos train.csv, val.csv y test.csv para el procesamiento del modelo

In [1]:
import pandas as pd
from google.colab import drive
import os

drive.mount('/content/drive')

path = "/content/drive/MyDrive/Tesis 2/Entrenamiento modelos NMT/Yine/train.csv"



print(os.listdir("/content/drive/MyDrive/Tesis 2/Entrenamiento modelos NMT/Yine"))

df = pd.read_csv(path)
df = df[["source", "target"]].dropna()

Mounted at /content/drive
['Relatos', 'Historias', 'corpus.csv', 'test.csv', 'train.csv', 'val.csv', '1. Limpieza de datos.ipynb', '2. Entrenamiento_mT5.ipynb', '2. Entrenamiento_NLLB', '2. Entrenamiento_mBART50.ipynb', '2_Entrenamiento_NLLB_v4.ipynb', '2_Entrenamiento_mT5_v4.ipynb', '2_Entrenamiento_mBART50_v4.ipynb']


In [2]:
#Limpiar texto

import re

def limpiar_texto(texto):
    return re.sub(r"[^\w\s-]", "", texto)

df["source_clean"] = df["source"].apply(limpiar_texto)
df["target_clean"] = df["target"].apply(limpiar_texto)

df.head()

,source,target,source_clean,target_clean
0,gike. netlu wagaga.,no. veo al guapo.,gike netlu wagaga,no veo al guapo
1,kagwuru wontapa.,vamos a ver la flor.,kagwuru wontapa,vamos a ver la flor
2,ga wa netatkalo yawo.,y vi al pelejo.,ga wa netatkalo yawo,y vi al pelejo
3,wane gyonakya.,ustedes cultivan allá.,wane gyonakya,ustedes cultivan allá
4,"kapiripa neta, ga wa kolyo, ga wa tora.","veo boquichico, y otro pescado, y corbina.",kapiripa neta ga wa kolyo ga wa tora,veo boquichico y otro pescado y corbina


In [3]:
##DEFINICIÓN DE SILABIFICADOR

VOCALES = "aeiou"
CONSONANTES_COMPUESTAS = ["ch", "sh", "ts"]
SUFIJOS = ["klu", "ru", "lu", "chri", "xlu"]

def dividir_palabra(palabra):
    silabas = []
    actual = ""
    i = 0

    while i < len(palabra):
        if i + 1 < len(palabra):
            par = palabra[i:i+2]
            if par in CONSONANTES_COMPUESTAS:
                actual += par
                i += 2
                continue

        letra = palabra[i]
        actual += letra

        if letra in VOCALES:
            silabas.append(actual)
            actual = ""

        i += 1

    if actual:
        silabas.append(actual)

    return silabas


def silabificar_yine(texto):
    palabras = texto.lower().split()
    resultado = []

    for palabra in palabras:
        if len(palabra) <= 4:
            resultado.append(palabra)
            continue

        partes = palabra.split("-")

        for parte in partes:
            sufijo_encontrado = False

            for sufijo in SUFIJOS:
                if parte.endswith(sufijo) and len(parte) > len(sufijo):
                    raiz = parte[:-len(sufijo)]
                    resultado.extend(dividir_palabra(raiz))
                    resultado.append(sufijo)
                    sufijo_encontrado = True
                    break

            if not sufijo_encontrado:
                resultado.extend(dividir_palabra(parte))

    return resultado

In [4]:
# =========================
# EXTRAER SUBWORDS Y SÍLABAS
# =========================
from collections import Counter
subword_counter = Counter()
for texto in df["source_clean"]:
    palabras = texto.split()
    for palabra in palabras:
        silabas = silabificar_yine(palabra)
        subword_counter.update(silabas)

print("SUBWORDS MÁS FRECUENTES:")
print(subword_counter.most_common(50))


SUBWORDS MÁS FRECUENTES:
[('wa', 183), ('ka', 172), ('ta', 160), ('ne', 138), ('ga', 138), ('gi', 123), ('pa', 89), ('ru', 81), ('na', 72), ('chi', 62), ('nu', 56), ('tka', 51), ('ya', 48), ('ge', 43), ('tu', 39), ('pi', 38), ('nka', 37), ('su', 30), ('ko', 29), ('wane', 28), ('ra', 28), ('po', 26), ('ni', 25), ('kle', 25), ('lu', 23), ('no', 23), ('wu', 23), ('yo', 23), ('tlo', 22), ('chri', 22), ('te', 21), ('we', 21), ('mama', 19), ('wiwi', 19), ('tya', 19), ('pga', 19), ('ma', 18), ('papa', 17), ('klu', 16), ('lo', 15), ('gita', 15), ('tna', 15), ('je', 15), ('tje', 15), ('ri', 14), ('tma', 14), ('tni', 14), ('pe', 14), ('t', 13), ('nro', 13)]


In [5]:
from datasets import Dataset
df_model = df[["source_clean", "target_clean"]].rename(
    columns={
        "source_clean": "source",
        "target_clean": "target"
    }
)
dataset = Dataset.from_pandas(df_model)
print(dataset)


Dataset({
    features: ['source', 'target'],
    num_rows: 372
})


In [6]:
#Tokenización

from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("facebook/mbart-large-50-many-to-many-mmt")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/529 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/649 [00:00<?, ?B/s]

In [7]:
##Ejemplo para verificar
example1 = dataset[0]["source"]
example2 = dataset[0]["target"]

print("Texto usado por el modelo:")
print(example1)
print(example2)

tokens = tokenizer(example1)
print(tokens["input_ids"])


Texto usado por el modelo:
gike netlu wagaga
no veo al guapo
[250004, 40130, 13, 2043, 822, 259, 90557, 2]


In [8]:
##Ejemplo de ambas lenguas

source_text = dataset[0]["source"]
target_text = dataset[0]["target"]

source_tokens = tokenizer(source_text)
target_tokens = tokenizer(target_text)

print("SOURCE IDS:", source_tokens["input_ids"])
print("TARGET IDS:", target_tokens["input_ids"])

SOURCE IDS: [250004, 40130, 13, 2043, 822, 259, 90557, 2]
TARGET IDS: [250004, 110, 173, 31, 144, 42348, 771, 2]


In [9]:
#CREACION DE FUNCION PARA TOKENIZAR TODO EL DATASET

def tokenizar_ejemplo(example):

    # texto de entrada (Yine silabificado)
    source = example["source"]

    # texto objetivo (español)
    target = example["target"]

    # tokenizamos input
    source_tokens = tokenizer(
        source,
        truncation=True,
        padding="max_length",
        max_length=64
    )

    # tokenizamos target
    target_tokens = tokenizer(
        target,
        truncation=True,
        padding="max_length",
        max_length=64
    )

    return {
        "input_ids": source_tokens["input_ids"],
        "attention_mask": source_tokens["attention_mask"],
        "labels": target_tokens["input_ids"]
    }

In [10]:
#Aplicamos la funcion pta tokenizar todo el dataset

tokenized_dataset = dataset.map(tokenizar_ejemplo)

Map:   0%|          | 0/372 [00:00<?, ? examples/s]

In [11]:
print(tokenized_dataset[0])

{'source': 'gike netlu wagaga', 'target': 'no veo al guapo', 'input_ids': [250004, 40130, 13, 2043, 822, 259, 90557, 2, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 'labels': [250004, 110, 173, 31, 144, 42348, 771, 2, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}


In [12]:
##Ejecutar solo una vez
tokenized_dataset = tokenized_dataset.remove_columns(["source", "target"])


In [13]:
print(tokenized_dataset)

Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 372
})


In [14]:

tokenized_dataset.set_format("torch")

In [15]:

from transformers import AutoModelForSeq2SeqLM

model = AutoModelForSeq2SeqLM.from_pretrained(
    "facebook/mbart-large-50-many-to-many-mmt"
)
# =========================
# REDIMENSIONAR EMBEDDINGS
# =========================
model.resize_token_embeddings(len(tokenizer))


model.safetensors:   0%|          | 0.00/2.44G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/261 [00:00<?, ?B/s]

MBartScaledWordEmbedding(250054, 1024, padding_idx=1)

In [17]:
# Parche de compatibilidad: torchvision en Colab puede no tener VideoReader
try:
    import torchvision.io
    if not hasattr(torchvision.io, 'VideoReader'):
        class _DummyVideoReader:
            pass
        torchvision.io.VideoReader = _DummyVideoReader
except ImportError:
    pass

tokenized_dataset.set_format("torch")


In [18]:
sample = tokenized_dataset[0]

# agregamos batch dimension
import torch

inputs = {
    "input_ids": sample["input_ids"].unsqueeze(0),
    "attention_mask": sample["attention_mask"].unsqueeze(0),
    "labels": sample["labels"].unsqueeze(0)
}

outputs = model(**inputs)
print(outputs)

Seq2SeqLMOutput(loss=tensor(12.0750, grad_fn=<NllLossBackward0>), logits=tensor([[[ -1.5615,  -1.5657,   0.7981,  ...,  -0.4024,   1.6374,  -1.2695],
         [ -1.5813,  -1.6891,  -0.9513,  ...,  -5.2027,  -4.0432,  -1.2735],
         [  3.5787,   3.3488,  12.9202,  ...,   3.6095,   2.6766,   3.4910],
         ...,
         [-16.2776, -15.7532,  -7.0333,  ..., -23.7390, -19.8218, -16.7932],
         [-16.4178, -15.8886,  -7.0657,  ..., -23.9519, -20.1263, -16.9613],
         [-16.2396, -15.7307,  -6.8129,  ..., -23.3077, -19.4890, -16.7657]]],
       grad_fn=<AddBackward0>), past_key_values=None, decoder_hidden_states=None, decoder_attentions=None, cross_attentions=None, encoder_last_hidden_state=tensor([[[ 6.3155e-03, -3.4070e-03, -9.9936e-04,  ..., -2.5436e-02,
           1.2149e-03,  4.7620e-03],
         [ 1.8847e-01,  9.1716e-01, -2.9140e-02,  ..., -1.5040e+00,
          -1.0083e-01, -5.6745e-01],
         [-2.6994e-01, -1.9055e-01, -2.1320e-01,  ..., -6.9282e-01,
          -1.12

In [19]:
##Código de entrenamiento (mBART50)

from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./results",              # dónde guarda modelo
    per_device_train_batch_size=1,       # cuántos ejemplos por paso
    num_train_epochs=3,                  # cuántas veces ve el dataset
    logging_dir="./logs",                # logs
    save_strategy="epoch",               # guarda cada epoch
)

In [20]:
#Creación del Trainer

from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset
)

In [21]:
import os

print(os.listdir("/content/drive/MyDrive/Tesis 2/Entrenamiento modelos NMT/Yine"))

['Relatos', 'Historias', 'corpus.csv', 'test.csv', 'train.csv', 'val.csv', '1. Limpieza de datos.ipynb', '2. Entrenamiento_mT5.ipynb', '2. Entrenamiento_NLLB', '2. Entrenamiento_mBART50.ipynb', '2_Entrenamiento_NLLB_v4.ipynb', '2_Entrenamiento_mT5_v4.ipynb', '2_Entrenamiento_mBART50_v4.ipynb']


In [23]:
trainer.train()

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results


wandb: Enter your choice: 3


wandb: You chose "Don't visualize my results"
wandb: Using W&B in offline mode.
wandb: W&B API key is configured. Use `wandb login --relogin` to force relogin


Step,Training Loss
500,0.859700
1000,0.165500


/usr/local/lib/python3.12/dist-packages/transformers/modeling_utils.py:3918: UserWarning: Moving the following attributes in the config to the generation config: {'max_length': 200, 'early_stopping': True, 'num_beams': 5}. You are seeing this warning because you've set generation parameters in the model config, as opposed to in the generation config.
  warnings.warn(


TrainOutput(global_step=1116, training_loss=0.4679821605750737, metrics={'train_runtime': 91.8157, 'train_samples_per_second': 12.155, 'train_steps_per_second': 12.155, 'total_flos': 151157301313536.0, 'train_loss': 0.4679821605750737, 'epoch': 3.0})

In [ ]:
#Entrenar

##SOLO EL PRIMER ENTRENAMIENTO
#trainer.train()

from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

# modelo entrenado
model = AutoModelForSeq2SeqLM.from_pretrained("/content/drive/MyDrive/Tesis 2/Entrenamiento modelos NMT/Yine/modelo_mbART_yine_v4_v4", local_files_only=True)

# tokenizer ORIGINAL (no el tuyo)
tokenizer = AutoTokenizer.from_pretrained("facebook/mbart-large-50-many-to-many-mmt", local_files_only=True)
# =========================
# AGREGAR TOKENS AL TOKENIZER
# =========================
tokens_filtrados = [token for token in nuevos_tokens if token not in tokenizer.get_vocab()]
print("Nuevos tokens reales:", len(tokens_filtrados))
tokens_agregados = tokenizer.add_tokens(tokens_filtrados)
print("Tokens agregados:", tokens_agregados)

# =========================
# REDIMENSIONAR EMBEDDINGS
# =========================
model.resize_token_embeddings(len(tokenizer))


In [24]:
#Guardar modelo

trainer.save_model("/content/drive/MyDrive/Tesis 2/Entrenamiento modelos NMT/Yine/modelo_mbART_yine_v4_v4")
tokenizer.save_pretrained("/content/drive/MyDrive/Tesis 2/Entrenamiento modelos NMT/Yine/modelo_mbART_yine_v4_v4")

('/content/drive/MyDrive/Tesis 2/Entrenamiento modelos NMT/Yine/modelo_mbART_yine_v4_v4/tokenizer_config.json',
 '/content/drive/MyDrive/Tesis 2/Entrenamiento modelos NMT/Yine/modelo_mbART_yine_v4_v4/special_tokens_map.json',
 '/content/drive/MyDrive/Tesis 2/Entrenamiento modelos NMT/Yine/modelo_mbART_yine_v4_v4/sentencepiece.bpe.model',
 '/content/drive/MyDrive/Tesis 2/Entrenamiento modelos NMT/Yine/modelo_mbART_yine_v4_v4/added_tokens.json',
 '/content/drive/MyDrive/Tesis 2/Entrenamiento modelos NMT/Yine/modelo_mbART_yine_v4_v4/tokenizer.json')

In [25]:
##Creación de función de traducción

def traducir(texto):
    inputs = tokenizer(texto, return_tensors="pt").to(model.device)

    outputs = model.generate(**inputs, max_length=50)

    return tokenizer.decode(outputs[0], skip_special_tokens=True)

***PRUEBA CON VAL.CSV***

In [26]:
# Cargar val.csv y definir las variables faltantes
path_val = "/content/drive/MyDrive/Tesis 2/Entrenamiento modelos NMT/Yine/val.csv"
df_val = pd.read_csv(path_val)
df_val = df_val[["source", "target"]].dropna()

# Preprocesamiento de val
df_val["source_clean"] = df_val["source"].apply(limpiar_texto)
df_val["target_clean"] = df_val["target"].apply(limpiar_texto)

# Crear Dataset de validación
from datasets import Dataset
df_val_model = df_val[["source_clean", "target_clean"]].rename(
    columns={
        "source_clean": "source",
        "target_clean": "target"
    }
)
val_dataset = Dataset.from_pandas(df_val_model)
print("Dataset de validación cargado con éxito:", val_dataset)


Dataset de validación cargado con éxito: Dataset({
    features: ['source', 'target'],
    num_rows: 46
})


In [27]:
#Probar ejemplos

for i in range(5):
    print("INPUT:", df_val.iloc[i]["source_clean"])
    print("REAL:", val_dataset[i]["target"])
    print("PRED:", traducir(df_val.iloc[i]["source_clean"]))
    print("-----")

INPUT: katu ganunrotanu
REAL: con quién va a casarse
PRED: quién llega
-----
INPUT: ga wa wiwi tora nika
REAL: y mi hermanito come corbina
PRED: mi hermanito come plátanos
-----
INPUT: gi piklu papa
REAL: mi papá no le tenía miedo
PRED: mi papá no tiene miedo
-----
INPUT: wuya gi patewatanutka
REAL: anda no tengas más vergüenza
PRED: hemos tomado suficiente
-----
INPUT: gi ge ponro peta
REAL: no has visto al ciempiés
PRED: no ves al ciempiés
-----


In [28]:
#Creación de las métricas

##PASO 1 - Preparar predicciones

preds = []
refs = []

for i in range(len(val_dataset)):
    pred = traducir(df_val.iloc[i]["source_clean"])
    ref = val_dataset[i]["target"]

    preds.append(pred)
    refs.append([ref])  # BLEU necesita lista de listas


In [29]:
#BLEU

import evaluate

bleu = evaluate.load("bleu")

bleu_result = bleu.compute(predictions=preds, references=refs)

print("BLEU:", bleu_result)

BLEU: {'bleu': 0.23345346097473396, 'precisions': [0.4724770642201835, 0.26744186046511625, 0.18253968253968253, 0.16049382716049382], 'brevity_penalty': 0.9464417321993595, 'length_ratio': 0.9478260869565217, 'translation_length': 218, 'reference_length': 230}


In [30]:
#CHRF

chrf = evaluate.load("chrf")

chrf_result = chrf.compute(predictions=preds, references=refs)

print("ChrF:", chrf_result)

ChrF: {'score': 38.23305788884938, 'char_order': 6, 'word_order': 0, 'beta': 2}


In [31]:
#COMET

from comet import download_model, load_from_checkpoint

model_path = download_model("Unbabel/wmt22-comet-da")
comet_model = load_from_checkpoint(model_path)

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

LICENSE: 0.00B [00:00, ?B/s]

README.md: 0.00B [00:00, ?B/s]

hparams.yaml:   0%|          | 0.00/567 [00:00<?, ?B/s]

.gitattributes: 0.00B [00:00, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/616 [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/pytorch_lightning/core/saving.py:197: Found keys that are not in the model state dict but in the checkpoint: ['encoder.model.embeddings.position_ids']


In [32]:
##COMET - Evaluación

data = []

for i in range(len(val_dataset)):
    data.append({
        "src": df_val.iloc[i]["source_clean"],
        "mt": preds[i],
        "ref": val_dataset[i]["target"]
    })

scores = comet_model.predict(data, batch_size=8)

print("COMET:", scores["system_score"])

INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.utilities.rank_zero:💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
INFO:pytorch_lightning.utilities.rank_zero:💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
INFO:pytorch_lightning.utilities.rank_zero:You are using a CUDA device ('NVIDIA RTX PRO 6000 Blackwell Server Edition') that has Tensor Cores. To properly utilize them, you should set `torch.set_float32_matmul_precision('medium' | 'high')` which will trade-off precision for performance. For more details, read ht

COMET: 0.6645875294571337


In [33]:
# =========================
# MÉTRICA METEOR (VALIDACIÓN)
# =========================
try:
    import evaluate
    meteor = evaluate.load("meteor")
    meteor_result = meteor.compute(predictions=preds, references=[r[0] for r in refs])
    print("METEOR (Validación):", meteor_result)
except Exception as e:
    print("Error al calcular METEOR en validación:", e)


[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...


METEOR (Validación): {'meteor': 0.34801416697338516}


In [34]:
# =========================
# MÉTRICA BERTSCORE (VALIDACIÓN)
# =========================
try:
    import evaluate
    bertscore = evaluate.load("bertscore")
    bertscore_result = bertscore.compute(predictions=preds, references=[r[0] for r in refs], lang="es")
    print("BERTScore Precision (Validación - Promedio):", sum(bertscore_result["precision"]) / len(bertscore_result["precision"]))
    print("BERTScore Recall (Validación - Promedio):", sum(bertscore_result["recall"]) / len(bertscore_result["recall"]))
    print("BERTScore F1 (Validación - Promedio):", sum(bertscore_result["f1"]) / len(bertscore_result["f1"]))
except Exception as e:
    print("Error al calcular BERTScore en validación:", e)


tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/625 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/714M [00:00<?, ?B/s]

BERTScore Precision (Validación - Promedio): 0.8071105259916057
BERTScore Recall (Validación - Promedio): 0.8077767620915952
BERTScore F1 (Validación - Promedio): 0.8071729787017988


In [35]:
## REENTRENAR EL MODELO, PARA REVISION DE MEJORA
# ==============================================================================
# CAMBIOS PARA EL REENTRENAMIENTO INCREMENTAL:
# 1. learning_rate = 3e-5 (Se reduce a 3e-5 en comparación con el valor inicial de 5e-5)
#    para realizar un ajuste fino más conservador y evitar el olvido catastrófico de los pesos
#    ya optimizados en la fase 1.
# 2. num_train_epochs = 3 (Se mantiene en 3 épocas para el refinamiento adicional).
# ==============================================================================

training_args = TrainingArguments(
    output_dir="./results",
    per_device_train_batch_size=1,
    num_train_epochs=3,
    learning_rate=3e-5,
    save_strategy="epoch"
)


In [36]:
##Limpiamos el entorno

import torch
torch.cuda.empty_cache()

In [37]:
import torch
import gc

del model
del trainer

gc.collect()
torch.cuda.empty_cache()

In [40]:
from transformers import AutoModelForSeq2SeqLM
import os

# ==============================================================================
# REENTRENAMIENTO INCREMENTAL RESILIENTE:
# Busca los pesos de la Fase 1 en Google Drive (v4 o original) para continuar el aprendizaje.
# Si no encuentra ninguno, cae al modelo base de Hugging Face por seguridad.
# ==============================================================================
path_v4 = "/content/drive/MyDrive/Tesis 2/Entrenamiento modelos NMT/Yine/modelo_mbART_yine_v4_v4"
path_orig = "/content/drive/MyDrive/Tesis 2/Entrenamiento modelos NMT/Yine/modelo_mbART_yine"

if os.path.exists(path_v4):
    print(f"[INFO] Cargando pesos de Fase 1 (v4) para reentrenamiento: {path_v4}")
    model = AutoModelForSeq2SeqLM.from_pretrained(path_v4, local_files_only=True)
elif os.path.exists(path_orig):
    print(f"[INFO] Cargando pesos de Fase 1 (original) para reentrenamiento: {path_orig}")
    model = AutoModelForSeq2SeqLM.from_pretrained(path_orig, local_files_only=True)
else:
    print("[ADVERTENCIA] No se encontró modelo previo. Cargando modelo base desde cero...")
    model = AutoModelForSeq2SeqLM.from_pretrained("facebook/mbart-large-50-many-to-many-mmt")

model.to("cuda")
# =========================
# REDIMENSIONAR EMBEDDINGS
# =========================
model.resize_token_embeddings(len(tokenizer))


[INFO] Cargando pesos de Fase 1 (v4) para reentrenamiento: /content/drive/MyDrive/Tesis 2/Entrenamiento modelos NMT/Yine/modelo_mbART_yine_v4_v4


MBartScaledWordEmbedding(250054, 1024, padding_idx=1)

In [41]:
##Creamos del trainer despues de limpiar memoria

from transformers import Trainer
from transformers import TrainingArguments

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset
)

In [42]:
#Realizamos el entrenamiento despues de los siguientes cambios de parametros

trainer.train()

Step,Training Loss
500,0.061100
1000,0.020700


TrainOutput(global_step=1116, training_loss=0.03776541886363833, metrics={'train_runtime': 88.0972, 'train_samples_per_second': 12.668, 'train_steps_per_second': 12.668, 'total_flos': 151157301313536.0, 'train_loss': 0.03776541886363833, 'epoch': 3.0})

In [43]:
##Guardar 2da version

trainer.save_model("/content/drive/MyDrive/Tesis 2/Entrenamiento modelos NMT/Yine/modelo_mbART_yine_v4_retrained")
tokenizer.save_pretrained("/content/drive/MyDrive/Tesis 2/Entrenamiento modelos NMT/Yine/modelo_mbART_yine_v4_retrained")

('/content/drive/MyDrive/Tesis 2/Entrenamiento modelos NMT/Yine/modelo_mbART_yine_v4_retrained/tokenizer_config.json',
 '/content/drive/MyDrive/Tesis 2/Entrenamiento modelos NMT/Yine/modelo_mbART_yine_v4_retrained/special_tokens_map.json',
 '/content/drive/MyDrive/Tesis 2/Entrenamiento modelos NMT/Yine/modelo_mbART_yine_v4_retrained/sentencepiece.bpe.model',
 '/content/drive/MyDrive/Tesis 2/Entrenamiento modelos NMT/Yine/modelo_mbART_yine_v4_retrained/added_tokens.json',
 '/content/drive/MyDrive/Tesis 2/Entrenamiento modelos NMT/Yine/modelo_mbART_yine_v4_retrained/tokenizer.json')

In [49]:
# Carga y preparación del dataset de validación para evaluar el modelo después del reentrenamiento
print("Generando predicciones sobre el conjunto de validación con el modelo reentrenado...")

#Probar ejemplos

for i in range(5):
    print("INPUT:", df_val.iloc[i]["source_clean"])
    print("REAL:", val_dataset[i]["target"])
    print("PRED:", traducir(df_val.iloc[i]["source_clean"]))
    print("-----")


Generando predicciones sobre el conjunto de validación con el modelo reentrenado...
INPUT: katu ganunrotanu
REAL: con quién va a casarse
PRED: quién llega a la casa
-----
INPUT: ga wa wiwi tora nika
REAL: y mi hermanito come corbina
PRED: y mi hermanito come corbina
-----
INPUT: gi piklu papa
REAL: mi papá no le tenía miedo
PRED: mi papá no tiene miedo
-----
INPUT: wuya gi patewatanutka
REAL: anda no tengas más vergüenza
PRED: no estás enfermo tú
-----
INPUT: gi ge ponro peta
REAL: no has visto al ciempiés
PRED: no has visto al ciempiés
-----


In [45]:
#Generar predicciones

preds = []
refs = []

for i in range(len(val_dataset)):
    pred = traducir(df_val.iloc[i]["source_clean"])
    ref = val_dataset[i]["target"]

    preds.append(pred)
    refs.append([ref])

In [46]:
##Aplicamos nuevamente las métricas


##BLEU

import evaluate

bleu = evaluate.load("bleu")
bleu.compute(predictions=preds, references=refs)


{'bleu': 0.24948548280199626,
 'precisions': [0.46835443037974683,
  0.28272251308900526,
  0.19310344827586207,
  0.15151515151515152],
 'brevity_penalty': 1.0,
 'length_ratio': 1.0304347826086957,
 'translation_length': 237,
 'reference_length': 230}

In [47]:
##CHRF

chrf = evaluate.load("chrf")

chrf_result = chrf.compute(predictions=preds, references=refs)

print("ChrF:", chrf_result)

ChrF: {'score': 42.91976885143839, 'char_order': 6, 'word_order': 0, 'beta': 2}


In [48]:
##COMET - Evaluación

from comet import download_model, load_from_checkpoint

model_path = download_model("Unbabel/wmt22-comet-da")
comet_model = load_from_checkpoint(model_path)

data = []

for i in range(len(val_dataset)):
    data.append({
        "src": df_val.iloc[i]["source_clean"],
        "mt": preds[i],
        "ref": val_dataset[i]["target"]
    })

scores = comet_model.predict(data, batch_size=8)

print("COMET:", scores["system_score"])

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

INFO:pytorch_lightning.utilities.migration.utils:Lightning automatically upgraded your loaded checkpoint from v1.8.3.post1 to v2.6.5. To apply the upgrade to your files permanently, run `python -m pytorch_lightning.utilities.upgrade_checkpoint ../root/.cache/huggingface/hub/models--Unbabel--wmt22-comet-da/snapshots/2760a223ac957f30acfb18c8aa649b01cf1d75f2/checkpoints/model.ckpt`
/usr/local/lib/python3.12/dist-packages/pytorch_lightning/core/saving.py:197: Found keys that are not in the model state dict but in the checkpoint: ['encoder.model.embeddings.position_ids']
INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.utilities.rank_zero:💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiment

COMET: 0.6596633850232415


In [50]:
# =========================
# MÉTRICA METEOR (VALIDACIÓN - REENTRENADO)
# =========================
try:
    import evaluate
    meteor = evaluate.load("meteor")
    meteor_result = meteor.compute(predictions=preds, references=[r[0] for r in refs])
    print("METEOR (Validación - Reentrenado):", meteor_result)
except Exception as e:
    print("Error al calcular METEOR en validación con reentrenamiento:", e)


METEOR (Validación - Reentrenado): {'meteor': 0.4082942443094478}


[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


In [51]:
# =========================
# MÉTRICA BERTSCORE (VALIDACIÓN - REENTRENADO)
# =========================
try:
    import evaluate
    bertscore = evaluate.load("bertscore")
    bertscore_result = bertscore.compute(predictions=preds, references=[r[0] for r in refs], lang="es")
    print("BERTScore Precision (Validación - Reentrenado - Promedio):", sum(bertscore_result["precision"]) / len(bertscore_result["precision"]))
    print("BERTScore Recall (Validación - Reentrenado - Promedio):", sum(bertscore_result["recall"]) / len(bertscore_result["recall"]))
    print("BERTScore F1 (Validación - Reentrenado - Promedio):", sum(bertscore_result["f1"]) / len(bertscore_result["f1"]))
except Exception as e:
    print("Error al calcular BERTScore en validación con reentrenamiento:", e)


BERTScore Precision (Validación - Reentrenado - Promedio): 0.8182660289432692
BERTScore Recall (Validación - Reentrenado - Promedio): 0.8186002386652905
BERTScore F1 (Validación - Reentrenado - Promedio): 0.8181916462338489


In [52]:
##PROBAMOS EL TEST.CSV

path_test = "/content/drive/MyDrive/Tesis 2/Entrenamiento modelos NMT/Yine/test.csv"

df_test = pd.read_csv(path_test)
df_test = df_test[["source", "target"]].dropna()

In [53]:
#Preprocesamiento de test
df_test["source_clean"] = df_test["source"].apply(limpiar_texto)
df_test["target_clean"] = df_test["target"].apply(limpiar_texto)


In [54]:
from datasets import Dataset

# ==============================================================================
# CREACIÓN DEL DATASET DE PRUEBA:
# Corregimos la NameError asegurando que se carga a partir de df_test (y no del df de entrenamiento).
# ==============================================================================
df_test_model = df_test[["source_clean", "target_clean"]].rename(
    columns={
        "source_clean": "source",
        "target_clean": "target"
    }
)
test_dataset = Dataset.from_pandas(df_test_model)
print("Dataset de prueba cargado con éxito:", test_dataset)


Dataset de prueba cargado con éxito: Dataset({
    features: ['source', 'target'],
    num_rows: 47
})


In [55]:
preds = []
refs = []

for i in range(len(test_dataset)):
    pred = traducir(df_test.iloc[i]["source_clean"])
    ref = test_dataset[i]["target"]

    preds.append(pred)
    refs.append([ref])

In [56]:
##BLEU

bleu.compute(predictions=preds, references=refs)

{'bleu': 0.17399583963054502,
 'precisions': [0.4144486692015209,
  0.25,
  0.14792899408284024,
  0.09016393442622951],
 'brevity_penalty': 0.9024323076535476,
 'length_ratio': 0.906896551724138,
 'translation_length': 263,
 'reference_length': 290}

In [57]:
##CHRF
chrf.compute(predictions=preds, references=refs)

{'score': 36.017732204408816, 'char_order': 6, 'word_order': 0, 'beta': 2}

In [58]:
##COMET - Evaluación

from comet import download_model, load_from_checkpoint

model_path = download_model("Unbabel/wmt22-comet-da")
comet_model = load_from_checkpoint(model_path)

data = []

for i in range(len(test_dataset)):
    data.append({
        "src": df_test.iloc[i]["source_clean"],
        "mt": preds[i],
        "ref": test_dataset[i]["target"]
    })

scores = comet_model.predict(data, batch_size=8)

print("COMET:", scores["system_score"])

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

INFO:pytorch_lightning.utilities.migration.utils:Lightning automatically upgraded your loaded checkpoint from v1.8.3.post1 to v2.6.5. To apply the upgrade to your files permanently, run `python -m pytorch_lightning.utilities.upgrade_checkpoint ../root/.cache/huggingface/hub/models--Unbabel--wmt22-comet-da/snapshots/2760a223ac957f30acfb18c8aa649b01cf1d75f2/checkpoints/model.ckpt`
/usr/local/lib/python3.12/dist-packages/pytorch_lightning/core/saving.py:197: Found keys that are not in the model state dict but in the checkpoint: ['encoder.model.embeddings.position_ids']
INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.utilities.rank_zero:💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiment

COMET: 0.6515306373859974


In [59]:
# =========================
# MÉTRICA METEOR
# =========================
try:
    import evaluate
    meteor = evaluate.load("meteor")
    meteor_result = meteor.compute(predictions=preds, references=[r[0] for r in refs])
    print("METEOR:", meteor_result)
except Exception as e:
    print("Error al calcular METEOR:", e)


METEOR: {'meteor': 0.3898303179764688}


[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


In [60]:
# =========================
# MÉTRICA BERTSCORE
# =========================
try:
    import evaluate
    bertscore = evaluate.load("bertscore")
    bertscore_result = bertscore.compute(predictions=preds, references=[r[0] for r in refs], lang="es")
    print("BERTScore Precision (Promedio):", sum(bertscore_result["precision"]) / len(bertscore_result["precision"]))
    print("BERTScore Recall (Promedio):", sum(bertscore_result["recall"]) / len(bertscore_result["recall"]))
    print("BERTScore F1 (Promedio):", sum(bertscore_result["f1"]) / len(bertscore_result["f1"]))
except Exception as e:
    print("Error al calcular BERTScore:", e)


BERTScore Precision (Promedio): 0.8012744213672395
BERTScore Recall (Promedio): 0.7987027523365426
BERTScore F1 (Promedio): 0.7996257188472342
